In [2]:
import sqlite3

print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.49.1


In [3]:
conn = sqlite3.connect("../database/retail.db")
cursor = conn.cursor()

print("Database connected successfully!")

Database connected successfully!


In [4]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""")

tables = cursor.fetchall()

print("Tables in the Database:")
for table in tables:
    print("-", table[0])

Tables in the Database:
- customers
- products
- orders


In [5]:
for table in ["customers", "products", "orders"]:
    cursor.execute("""
        SELECT sql
        FROM sqlite_master
        WHERE type = 'table' AND name = ?
    """, (table,))
    
    result = cursor.fetchone()
    
    print(f"\n{table.upper()}")
    print(result[0] if result else "Table not found")


CUSTOMERS
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    region TEXT NOT NULL,
    customer_segment TEXT NOT NULL
)

PRODUCTS
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL
)

ORDERS
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    order_date DATE NOT NULL,
    quantity INTEGER NOT NULL,
    revenue REAL NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
)


In [6]:
for table in ["customers", "products", "orders"]:
    cursor.execute(f"SELECT * FROM {table} LIMIT 3")
    
    print(f"\n{table.upper()}")
    for row in cursor.fetchall():
        print(row)


CUSTOMERS
(1, 'Customer_001', 'North', 'Corporate')
(2, 'Customer_002', 'West', 'Consumer')
(3, 'Customer_003', 'South', 'Consumer')

PRODUCTS
(1, 'Wireless Mouse', 'Electronics', 25.99)
(2, 'Mechanical Keyboard', 'Electronics', 79.99)
(3, 'USB-C Hub', 'Electronics', 39.99)

ORDERS
(1, 79, 25, '2024-05-08', 4, 63.96)
(2, 62, 26, '2024-10-28', 1, 89.99)
(3, 25, 10, '2024-04-17', 2, 199.98)


In [24]:
from langchain_community.utilities import SQLDatabase
from langchain_ollama import OllamaLLM
from langchain_experimental.sql import SQLDatabaseChain

In [25]:
from pathlib import Path

db_path = Path.cwd().parent / "database" / "retail.db"

print(db_path)
print(db_path.exists())

c:\Users\sknav\Downloads\AI-Data-Analyst\database\retail.db
True


In [26]:
db = SQLDatabase.from_uri(f"sqlite:///{db_path.resolve()}")

print("Database connected to LangChain successfully!")

Database connected to LangChain successfully!


In [27]:
print(db.get_usable_table_names())

['customers', 'orders', 'products']


In [28]:
print(db.get_table_info())


CREATE TABLE customers (
	customer_id INTEGER, 
	customer_name TEXT NOT NULL, 
	region TEXT NOT NULL, 
	customer_segment TEXT NOT NULL, 
	PRIMARY KEY (customer_id)
)

/*
3 rows from customers table:
customer_id	customer_name	region	customer_segment
1	Customer_001	North	Corporate
2	Customer_002	West	Consumer
3	Customer_003	South	Consumer
*/


CREATE TABLE orders (
	order_id INTEGER, 
	customer_id INTEGER NOT NULL, 
	product_id INTEGER NOT NULL, 
	order_date DATE NOT NULL, 
	quantity INTEGER NOT NULL, 
	revenue REAL NOT NULL, 
	PRIMARY KEY (order_id), 
	FOREIGN KEY(customer_id) REFERENCES customers (customer_id), 
	FOREIGN KEY(product_id) REFERENCES products (product_id)
)

/*
3 rows from orders table:
order_id	customer_id	product_id	order_date	quantity	revenue
1	79	25	2024-05-08	4	63.96
2	62	26	2024-10-28	1	89.99
3	25	10	2024-04-17	2	199.98
*/


CREATE TABLE products (
	product_id INTEGER, 
	product_name TEXT NOT NULL, 
	category TEXT NOT NULL, 
	price REAL NOT NULL, 
	PRIMARY KEY (pro

In [29]:
# Load Ollama model
# We use temperature=0 because we want deterministic, precise SQL, not creative writing
llm = OllamaLLM(
    model="codellama",
    temperature=0
)

In [30]:
# Create SQL chain
db_chain = SQLDatabaseChain.from_llm(
    llm=llm,
    db=db,
    verbose=True,
    return_direct=True
)

In [34]:
db_chain = SQLDatabaseChain.from_llm(
    llm=llm,
    db=db,
    verbose=True,
    return_sql=True
)

print("return_sql =", db_chain.return_sql)
print("return_direct =", db_chain.return_direct)

return_sql = True
return_direct = False


In [31]:
# User question
question = "Which product generated the highest revenue?"

In [35]:
question = "Which product generated the highest revenue?"

response = db_chain.invoke({
    "query": question
})

print(response)



> Entering new SQLDatabaseChain chain...
Which product generated the highest revenue?
SQLQuery:
> Finished chain.
{'query': 'Which product generated the highest revenue?', 'result': 'Question: Which product generated the highest revenue?\nSQLQuery:\n```\nSELECT p.product_name, SUM(o.revenue) AS total_revenue\nFROM orders o\nINNER JOIN products p ON o.product_id = p.product_id\nGROUP BY p.product_name\nORDER BY total_revenue DESC\nLIMIT 1;\n```'}


In [37]:
# Print final response
print("\nFinal Response:")
print(response)


Final Response:
{'query': 'Which product generated the highest revenue?', 'result': 'Question: Which product generated the highest revenue?\nSQLQuery:\n```\nSELECT p.product_name, SUM(o.revenue) AS total_revenue\nFROM orders o\nINNER JOIN products p ON o.product_id = p.product_id\nGROUP BY p.product_name\nORDER BY total_revenue DESC\nLIMIT 1;\n```'}
